# SALEO Demo

Import

In [ ]:
from __future__ import annotations

import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

import saleo
from mon import albumentations as A, image as I, Path

Setup environment

In [ ]:
current_dir = Path(os.getcwd())
root_dir    = current_dir.parents[0]
data_dir    = current_dir / "demo"
output_dir  = current_dir / "demo"

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Debug
print(f"Device: {device}")

Resolve I/O

In [ ]:
image_name = "993_UHD_LL"

image_file = (data_dir / f"{image_name}").image_file()
depth_file = (data_dir / f"{image_name}_depth").image_file()

# Load data
image = cv2.imread(str(image_file))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

depth = cv2.imread(str(depth_file), cv2.IMREAD_GRAYSCALE)
if depth.ndim == 2:
    depth = np.expand_dims(depth, axis=-1)

# Debug
print(f"Image shape: {image.shape}")
print(f"Depth shape: {depth.shape}")
assert image.shape[0:2] == depth.shape[0:2], "'image' and 'depth' must have the same resolution."

Preprocess

In [ ]:
# Normalize and convert to torch.Tensor
transform = A.Compose([
    A.NormalizeWithMask(normalization="min_max"),
    A.ToTensorV2(transpose_mask=True),
])
transformed = transform(image=image, mask=depth)

image_t = transformed["image"]
depth_t = transformed["mask"]

image_t = image_t.unsqueeze(0)
depth_t = depth_t.unsqueeze(0)

# Debug
print(f"Image shape: {image_t.shape}. Is normalized: {I.is_normalized(image_t)}.")
print(f"Depth shape: {depth_t.shape}. Is normalized: {I.is_normalized(depth_t)}.")

Process

In [ ]:
model   = saleo.saleo_b_siren(E=0.5, verbose=True)
outputs = model(image=image_t, depth=depth_t, save_debug=True)

Postprocess

In [ ]:
enhanced      = I.to_array(outputs["enhanced"])
image_v       = I.to_array(outputs["image_v"])
image_v_fixed = I.to_array(outputs["image_v_fixed"])
residual      = I.to_array(outputs["residual"])

Visualize

In [ ]:
plt.rcParams["figure.autolayout"] = True

plt.imshow(enhanced)
plt.axis("off")
plt.show()